# Huấn luyện LTX 2.3 LoRA với Colab Notebook

Notebook này hướng dẫn huấn luyện LoRA cho mô hình LTX 2.3 sử dụng thư viện `musubi-tuner`.
Tất cả công việc tính toán (Model, Dataset, Caching) sẽ được thực hiện trên Local SSD của Colab (`/content/workspace`) để đảm bảo tốc độ huấn luyện tối đa.
Google Drive sẽ chỉ đóng vai trò lưu trữ (tải model từ Drive nếu có, tải model về Drive nếu chưa có, và lưu kết quả LoRA cuối cùng).

**Lưu ý**: Khuyến nghị GPU có 24GB VRAM trở lên (chọn GPU A100 hoặc L4/T4 trên Colab).

## Bước 1: Khởi tạo Môi trường & Mount Google Drive

Gắn kết Google Drive để phục vụ việc sao lưu. Tải và cài đặt các thư viện cần thiết vào hệ thống local.

In [ ]:
#@title Khởi tạo môi trường
mount_drive = True #@param {type:"boolean"}
drive_path = "/content/drive/MyDrive/LTX23_Workspace" #@param {type:"string"}

import os
import shutil

WORKSPACE = "/content/workspace"
os.makedirs(WORKSPACE, exist_ok=True)

if mount_drive:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(drive_path, exist_ok=True)
    DRIVE_WORKSPACE = drive_path
    print(f"Đã mount Google Drive. Thư mục sao lưu: {DRIVE_WORKSPACE}")
else:
    DRIVE_WORKSPACE = None
    print("Không sử dụng Google Drive.")

print(f"Thư mục làm việc chính (Local SSD): {WORKSPACE}")

# Clone Repo
repo_dir = os.path.join(WORKSPACE, "musubi-tuner")
if not os.path.exists(repo_dir):
    !git clone https://github.com/pmhaidn/musubi-tuner.git {repo_dir}

os.chdir(repo_dir)

# Cài đặt dependencies (chỉ chạy cài đặt trong môi trường Colab hiện tại)
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -e .
!pip install ascii-magic matplotlib tensorboard accelerate huggingface_hub

## Bước 2: Tải Mô hình LTX 2.3 & Gemma

Hệ thống sẽ kiểm tra xem model đã có trên Google Drive chưa. Nếu có, nó sẽ sao chép tốc độ cao sang Local SSD (`/content/workspace/models`). Nếu chưa, hệ thống sẽ tải trực tiếp từ HuggingFace vào Drive, sau đó copy sang Local SSD.

In [ ]:
#@title Tải Model (Local SSD)
import os
import shutil
from huggingface_hub import hf_hub_download

local_model_dir = os.path.join(WORKSPACE, "models")
os.makedirs(local_model_dir, exist_ok=True)

ltx_filename = "ltx-2.3-22b-dev.safetensors"
gemma_filename = "gemma_3_12B_it_fp8_e4m3fn.safetensors"

local_ltx_path = os.path.join(local_model_dir, ltx_filename)
local_gemma_path = os.path.join(local_model_dir, os.path.basename(gemma_filename))

drive_model_dir = os.path.join(DRIVE_WORKSPACE, "models") if DRIVE_WORKSPACE else local_model_dir
os.makedirs(drive_model_dir, exist_ok=True)

drive_ltx_path = os.path.join(drive_model_dir, ltx_filename)
drive_gemma_path = os.path.join(drive_model_dir, os.path.basename(gemma_filename))

# Xử lý LTX 2.3
if not os.path.exists(local_ltx_path):
    if os.path.exists(drive_ltx_path):
        print("Đang copy LTX 2.3 từ Drive sang Local SSD...")
        shutil.copy2(drive_ltx_path, local_ltx_path)
    else:
        print("Đang tải LTX 2.3 Checkpoint từ HuggingFace...")
        downloaded = hf_hub_download(repo_id="Lightricks/LTX-2.3", filename=ltx_filename, local_dir=drive_model_dir)
        print("Đang copy LTX 2.3 sang Local SSD...")
        if downloaded != local_ltx_path: # Phòng trường hợp không dùng Drive
            shutil.copy2(downloaded, local_ltx_path)
else:
    print("LTX 2.3 Checkpoint đã có trên Local SSD.")

# Xử lý Gemma
if not os.path.exists(local_gemma_path):
    if os.path.exists(drive_gemma_path):
        print("Đang copy Gemma từ Drive sang Local SSD...")
        shutil.copy2(drive_gemma_path, local_gemma_path)
    else:
        print("Đang tải Gemma FP8 Text Encoder từ HuggingFace...")
        downloaded = hf_hub_download(repo_id="GitMylo/LTX-2-comfy_gemma_fp8_e4m3fn", filename=gemma_filename, local_dir=drive_model_dir)
        print("Đang copy Gemma sang Local SSD...")
        if downloaded != local_gemma_path:
            shutil.copy2(downloaded, local_gemma_path)
else:
    print("Gemma FP8 Text Encoder đã có trên Local SSD.")

print("Models sẵn sàng tại Local SSD:", local_model_dir)

## Bước 3: Upload Dataset & Cấu hình TOML

Chọn `upload_dataset = True` để hiện thị nút tải lên (Upload) các file video (.mp4) và text (.txt) trực tiếp từ máy tính của bạn.
Hoặc bỏ chọn `upload_dataset` và điền đường dẫn `drive_dataset_path` để tự động copy toàn bộ dataset từ Drive sang Local SSD.

In [ ]:
#@title Upload/Chuẩn bị Dataset
upload_dataset = False #@param {type:"boolean"}
drive_dataset_path = "/content/drive/MyDrive/LTX23_Workspace/dataset/videos" #@param {type:"string"}
resolution_width = 768 #@param {type:"integer"}
resolution_height = 512 #@param {type:"integer"}
target_fps = 25 #@param {type:"integer"}
batch_size = 1 #@param {type:"integer"}

import os
import shutil

local_dataset_dir = os.path.join(WORKSPACE, "dataset/videos")
os.makedirs(local_dataset_dir, exist_ok=True)

local_cache_dir = os.path.join(WORKSPACE, "dataset_cache")
os.makedirs(local_cache_dir, exist_ok=True)

if upload_dataset:
    from google.colab import files
    print("Vui lòng nhấn 'Choose Files' để tải lên các file video và caption (.txt).")
    # Đổi thư mục tạm thời để file được lưu thẳng vào dataset
    os.chdir(local_dataset_dir)
    uploaded = files.upload()
    os.chdir(repo_dir) # Quay lại thư mục repo
    print(f"Đã upload {len(uploaded)} file vào {local_dataset_dir}")
else:
    if os.path.exists(drive_dataset_path):
        print(f"Đang copy dataset từ Drive ({drive_dataset_path}) sang Local SSD...")
        # Copy nội dung từ Drive sang Local
        for filename in os.listdir(drive_dataset_path):
            source = os.path.join(drive_dataset_path, filename)
            destination = os.path.join(local_dataset_dir, filename)
            if os.path.isfile(source):
                shutil.copy2(source, destination)
        print("Đã copy dataset thành công.")
    else:
        print(f"CẢNH BÁO: Không tìm thấy thư mục {drive_dataset_path} trên Drive!")

toml_content = f"""[general]
resolution = [{resolution_width}, {resolution_height}]
caption_extension = ".txt"
batch_size = {batch_size}
enable_bucket = true

[[datasets]]
video_directory = "{local_dataset_dir}"
cache_directory = "{local_cache_dir}"
target_frames = [1, 17, 33]
target_fps = {target_fps}
"""

toml_path = os.path.join(repo_dir, "dataset.toml")
with open(toml_path, "w") as f:
    f.write(toml_content)

print(f"Đã tạo cấu hình {toml_path} trỏ tới thư mục Local SSD.")

## Bước 4: Pre-caching

Latents và Text Embeddings sẽ được mã hóa và lưu trữ trực tiếp trên Local SSD để đạt tốc độ truy xuất nhanh nhất khi huấn luyện.

In [ ]:
#@title Cache Latents & Text Encoder Outputs (Local SSD)
vae_chunk_size = 16 #@param {type:"integer"}

import subprocess

print("1. Caching Latents (Tốc độ cao trên Local SSD)...")
subprocess.run([
    "python", "ltx2_cache_latents.py",
    "--dataset_config", "dataset.toml",
    "--ltx2_checkpoint", local_ltx_path,
    "--device", "cuda",
    "--vae_dtype", "bf16",
    "--vae_chunk_size", str(vae_chunk_size),
    "--ltx2_mode", "video"
], check=True)

print("2. Caching Text Encoder Outputs (Tốc độ cao trên Local SSD)...")
subprocess.run([
    "python", "ltx2_cache_text_encoder_outputs.py",
    "--dataset_config", "dataset.toml",
    "--ltx2_checkpoint", local_ltx_path,
    "--gemma_safetensors", local_gemma_path,
    "--device", "cuda",
    "--mixed_precision", "bf16",
    "--ltx2_mode", "video",
    "--batch_size", "1"
], check=True)
print("Hoàn tất Pre-caching!")

## Bước 5: Huấn luyện LoRA & Sao lưu kết quả

Quá trình huấn luyện diễn ra trên Local SSD. Sau khi hoàn tất, kết quả LoRA sẽ tự động được sao lưu vào Google Drive.

In [ ]:
#@title Tham số Huấn luyện
learning_rate = "1e-4" #@param {type:"string"}
max_train_epochs = 10 #@param {type:"integer"}
save_every_n_epochs = 5 #@param {type:"integer"}
network_dim = 32 #@param {type:"integer"}
network_alpha = 32 #@param {type:"integer"}
blocks_to_swap = 30 #@param {type:"integer"}
output_name = "ltx23_lora" #@param {type:"string"}

local_output_dir = os.path.join(WORKSPACE, "output")
os.makedirs(local_output_dir, exist_ok=True)

train_cmd = [
    "accelerate", "launch", "--num_cpu_threads_per_process", "1", "ltx2_train_network.py",
    "--mixed_precision", "bf16",
    "--dataset_config", "dataset.toml",
    "--ltx2_checkpoint", local_ltx_path,
    "--ltx_version", "2.3",
    "--ltx_version_check_mode", "warn",
    "--ltx2_mode", "video",
    "--fp8_base", "--fp8_scaled",
    "--blocks_to_swap", str(blocks_to_swap),
    "--use_pinned_memory_for_block_swap",
    "--gradient_checkpointing",
    "--gradient_checkpointing_cpu_offload",
    "--sdpa",
    "--learning_rate", learning_rate,
    "--network_module", "networks.lora_ltx2",
    "--network_dim", str(network_dim),
    "--network_alpha", str(network_alpha),
    "--timestep_sampling", "shifted_logit_normal",
    "--output_dir", local_output_dir,
    "--output_name", output_name,
    "--max_train_epochs", str(max_train_epochs),
    "--save_every_n_epochs", str(save_every_n_epochs)
]

import subprocess
print("Bắt đầu huấn luyện trên Local SSD...")
subprocess.run(train_cmd, check=True)
print(f"Huấn luyện hoàn tất. LoRA được lưu tại: {local_output_dir}")

if DRIVE_WORKSPACE:
    drive_output_dir = os.path.join(DRIVE_WORKSPACE, "output")
    os.makedirs(drive_output_dir, exist_ok=True)
    print(f"Đang sao lưu các file LoRA sang Drive: {drive_output_dir}")
    for filename in os.listdir(local_output_dir):
        if filename.endswith(".safetensors") or filename.endswith(".json"):
            source = os.path.join(local_output_dir, filename)
            destination = os.path.join(drive_output_dir, filename)
            shutil.copy2(source, destination)
            print(f"Đã copy: {filename}")
    print("Sao lưu hoàn tất!")